In [12]:
# ── CELL 1: Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [ ]:
# ── CELL 2: Load Data ────────────────────────────────────────────
from google.colab import files
files.upload()
df = pd.read_csv('/content/tmdb_5000_movies.csv')

In [ ]:
# ── CELL 3: Quick Info ───────────────────────────────────────────
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

In [ ]:
# ── CELL 4: Basic Stats ──────────────────────────────────────────
df.describe()


In [ ]:
# ── CELL 5: Cleaning ─────────────────────────────────────────────
# Zero budget/revenue means unknown
df["budget"]  = df["budget"].replace(0, np.nan)
df["revenue"] = df["revenue"].replace(0, np.nan)

# Fix dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

# Keep only released movies with enough votes
df = df[df["status"] == "Released"]
df = df[df["vote_count"] >= 10]

# New columns
df["profit"] = df["revenue"] - df["budget"]
df["roi"]    = df["profit"] / df["budget"]

print("Done. Shape:", df.shape)
df[["title", "vote_average", "budget", "revenue", "profit"]].head()


In [ ]:
# ── CELL 6: Vote Average Distribution ───────────────────────────
plt.figure(figsize=(8, 4))
plt.hist(df["vote_average"], bins=30, color="steelblue", edgecolor="white")
plt.axvline(df["vote_average"].mean(), color="red", ls="--", label=f"Mean = {df['vote_average'].mean():.2f}")
plt.title("Vote Average Distribution")
plt.xlabel("Vote Average")
plt.legend()
plt.show()

In [ ]:
 #── CELL 7: Top 10 Genres ────────────────────────────────────────
import ast

def get_genres(val):
    try:
        return [d["name"] for d in ast.literal_eval(val)]
    except:
        return []

df["genres_list"] = df["genres"].apply(get_genres)

from collections import Counter
genre_counts = Counter(g for lst in df["genres_list"] for g in lst)
genre_df = pd.DataFrame(genre_counts.most_common(10), columns=["Genre", "Count"])

plt.figure(figsize=(8, 5))
sns.barplot(data=genre_df, x="Count", y="Genre", palette="Blues_r")
plt.title("Top 10 Genres")
plt.show()


In [ ]:
# ── CELL 8: Budget vs Revenue ────────────────────────────────────
fin = df.dropna(subset=["budget", "revenue"])

plt.figure(figsize=(7, 5))
plt.scatter(np.log1p(fin["budget"]), np.log1p(fin["revenue"]), alpha=0.4, s=15, color="steelblue")
plt.xlabel("log(Budget)")
plt.ylabel("log(Revenue)")
plt.title("Budget vs Revenue")
plt.show()


In [ ]:
# ── CELL 9: Movies per Year ──────────────────────────────────────
year_counts = df[df["release_year"].between(1980, 2017)]["release_year"].value_counts().sort_index()

plt.figure(figsize=(12, 4))
plt.bar(year_counts.index, year_counts.values, color="coral", width=0.8)
plt.title("Movies Released per Year")
plt.xlabel("Year")
plt.show()

In [ ]:
 #── CELL 10: Correlation Heatmap ─────────────────────────────────
cols = ["vote_average", "vote_count", "popularity", "budget", "revenue", "runtime"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# ── CELL 11: Top 10 Highest Rated Movies ────────────────────────
top10 = df[df["vote_count"] >= 100].nlargest(10, "vote_average")[["title", "vote_average"]]

plt.figure(figsize=(8, 5))
sns.barplot(data=top10, x="vote_average", y="title", palette="viridis")
plt.title("Top 10 Highest Rated Movies")
plt.xlabel("Vote Average")
plt.show()
